In [ ]:
import pandas as pd
import psycopg2
import DATABASE_CONFIG

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG.DB_NAME,
    user=DATABASE_CONFIG.DB_USER,
    password=DATABASE_CONFIG.DB_PASSWORD,
    host=DATABASE_CONFIG.DB_HOST,
    port=DATABASE_CONFIG.DB_PORT
)

cursor = conn.cursor()

In [98]:
def normalizar_numero(valor):
    if not isinstance(valor, str):
        valor = str(valor)
    return float(valor.replace('.', '').replace(',', '.'))

In [ ]:
df = pd.read_csv('../datasets/UnidadesConservacao.csv', sep=';')

df = df.rename(columns={
    'Nome da UC': 'nome',
    'Esfera Administrativa': 'esfera_administrativa',
    'Categoria de Manejo': 'categoria',
    'Grupo': 'tipo',
    'Informações Gerais': 'id_estado',
    'Ano de Criação': 'id_categoria_unidade_conservacao',
    'Amazônia': 'area_amazonia_ha',
})
df = df[['nome', 'UF', 'categoria', 'esfera_administrativa', 'tipo', 'area_amazonia_ha', 'id_estado']]
df['area_amazonia_ha'] = df['area_amazonia_ha'].apply(normalizar_numero)
uf_amazonia_legal = ['AC', 'AP', 'AM', 'MA', 'MT', 'PA', 'RO', 'RR', 'TO']

for index, row in df.iterrows():
    df.at[index, 'nome'] = row['nome'].title()
    uf = row['UF']

    if uf not in uf_amazonia_legal:
        df.drop(index, inplace=True)
        continue
    else:
        cursor.execute("""
            SELECT id_estado FROM estado WHERE uf = %s
        """, (uf,))
        estado_id = cursor.fetchone()[0]

    df.at[index, 'id_estado'] = estado_id

df.drop(columns=['UF'], inplace=True)
data = list(df.itertuples(index=False, name=None))
print(data)


[('Reserva Particular Do Patrimônio Natural Fazenda Estância Dorochê', 'Reserva Particular do Patrimônio Natural', 'Federal', 'Uso Sustentável', 0.0, 5.0), ('Reserva Particular Do Patrimônio Natural Jubran', 'Reserva Particular do Patrimônio Natural', 'Federal', 'Uso Sustentável', 0.0, 5.0), ('Reserva Particular Do Patrimônio Natural Estância Ecológica Sesc - Pantanal', 'Reserva Particular do Patrimônio Natural', 'Federal', 'Uso Sustentável', 0.0, 5.0), ('Reserva Particular Do Patrimônio Natural Estância Ecológica Sesc - Pantanal', 'Reserva Particular do Patrimônio Natural', 'Federal', 'Uso Sustentável', 0.0, 5.0), ('Estação Ecológica De Taiamã', 'Estação Ecológica', 'Federal', 'Proteção Integral', 0.0, 5.0), ('Parque Estadual Do Guirá', 'Parque', 'Estadual', 'Proteção Integral', 0.0, 5.0), ('Parque Estadual Encontro Das Águas', 'Parque', 'Estadual', 'Proteção Integral', 0.0, 5.0), ('Parque Estadual Marinho Do Parcel De Manuel Luís', 'Parque', 'Estadual', 'Proteção Integral', 0.0, 4.0)

In [102]:
from psycopg2.extras import execute_values

query = """
    INSERT INTO unidade_conservacao
    (nome, categoria, esfera_administrativa, tipo, area_amazonia_ha, id_estado)
    VALUES %s
    ON CONFLICT (nome) DO NOTHING
"""
execute_values(cursor, query, data)
# Finaliza
conn.commit()

In [108]:
command = """
SELECT id_cidade, nome_normalizado FROM cidade
"""
cursor.execute(command)
rows_cidade = cursor.fetchall()
ids_cidade = {nome: ids_cidade for ids_cidade, nome in rows_cidade}

print(ids_cidade)

{"alta floresta d'oeste": 672, 'alto alegre dos parecis': 673, 'alto paraiso': 674, "alvorada d'oeste": 675, 'ariquemes': 676, 'buritis': 677, 'cabixi': 678, 'cacaulandia': 679, 'cacoal': 680, 'campo novo de rondonia': 681, 'candeias do jamari': 682, 'castanheiras': 683, 'cerejeiras': 684, 'chupinguaia': 685, 'colorado do oeste': 686, 'corumbiara': 687, 'costa marques': 688, 'cujubim': 689, "espigao d'oeste": 690, 'governador jorge teixeira': 691, 'guajara-mirim': 692, 'itapua do oeste': 693, 'jaru': 694, 'ji-parana': 695, "machadinho d'oeste": 696, 'ministro andreazza': 697, 'mirante da serra': 698, 'monte negro': 699, "nova brasilandia d'oeste": 700, 'nova mamore': 701, 'nova uniao': 702, 'novo horizonte do oeste': 703, 'ouro preto do oeste': 704, 'parecis': 705, 'pimenta bueno': 706, 'pimenteiras do oeste': 707, 'porto velho': 708, 'presidente medici': 1273, 'primavera de rondonia': 710, 'rio crespo': 711, 'rolim de moura': 712, "santa luzia d'oeste": 713, "sao felipe d'oeste": 714,

In [104]:
import unicodedata

def normalizar(texto):
    unaccent = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return unaccent.lower()

In [107]:
def normalizar_lista_cidades_abrangidas(lista):
    cidades = lista.split("-")
    cidades = [normalizar(cidade[:-5].strip()) for cidade in cidades]
    return cidades

In [106]:
def buscar_unidade_conservacao(cursor, nome):
    command = """
    SELECT id_unidade_conservacao FROM unidade_conservacao
    WHERE nome = %s
    """
    cursor.execute(command, (nome,))
    id_unidade_conservacao = cursor.fetchone()
    return id_unidade_conservacao[0] if id_unidade_conservacao else None

In [57]:
def buscar_cidade(nome_busca):
    for nome_cadastro, id_cadastro in ids_cidade.items():
        if nome_cadastro == nome_busca:
            return id_cadastro

        nome_cadastro_lista = nome_cadastro.split()
        nome_busca_lista = nome_busca.split()
        if len(nome_cadastro_lista) != len(nome_busca_lista):
            continue

        if all(
            (parte_nome_cadastro == parte_nome_busca)
            or (
                (parte_nome_cadastro.endswith(".") or parte_nome_busca.endswith("."))
                and parte_nome_cadastro[0] == parte_nome_busca[0]
            )
            for parte_nome_cadastro, parte_nome_busca in zip(
                nome_cadastro_lista, nome_busca_lista
            )
        ):
            return id_cadastro

In [94]:
def buscar_estado(cursor, uf):
    command = """
    SELECT id_estado FROM estado
    WHERE uf = %s
    """
    cursor.execute(command, (uf,))
    id_estado = cursor.fetchone()
    return id_estado[0] if id_estado else None

In [121]:
def inserir_relacao_cidade_unidade_conservacao(cursor, id_cidade, id_unidade_conservacao):
    command = """
    INSERT INTO cidade_unidade_conservacao VALUES
    (%s, %s);
    """
    cursor.execute(command, (id_cidade, id_unidade_conservacao))

df = pd.read_csv('../datasets/UnidadesConservacao.csv', sep=';')
df = df.rename(columns={
    'Nome da UC': 'nome',
    'Municípios Abrangidos': 'cidades',
})
df = df[['nome', 'cidades']]

for _, row in df.iterrows():
    nome = row['nome'].title()
    cidades = row['cidades']

    id_unidade_conservacao = buscar_unidade_conservacao(cursor, nome)
    if id_unidade_conservacao is None:
        continue

    lista_cidades = normalizar_lista_cidades_abrangidas(cidades)
    for i in range(len(lista_cidades)):
        cidade = lista_cidades[i]
        id_cidade = buscar_cidade(cidade)
        if id_cidade is None:
            continue

        inserir_relacao_cidade_unidade_conservacao(cursor, id_cidade, id_unidade_conservacao)
    
conn.commit()

In [ ]:
conn.rollback()

In [122]:
cursor.close()
conn.close()
# Close the connection